# Phase 1 VLM + Quantum + CFD Workbook

This notebook supports two dataset modes:
- **Real NASA/NACA tabular files** when available locally.
- **Synthetic NACA-inspired fallback** when no real dataset is found.

All paths normalize to the same node schema used by the app backend:
`node_id, span_index, chord_index, x, y, z, lift, drag, cp, gamma`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')

In [ ]:
def naca_4digit_thickness(x, t=0.12):
    # Standard NACA thickness law for synthetic geometry shape
    return 5 * t * (0.2969*np.sqrt(np.maximum(x, 1e-8)) - 0.1260*x - 0.3516*x**2 + 0.2843*x**3 - 0.1015*x**4)

def generate_vlm_nodes(n_span=12, n_chord=18, span=1.6, chord=0.35, alpha_deg=5.0, velocity=72.0, rho=1.225):
    q_inf = 0.5 * rho * velocity**2
    x_positions = np.linspace(0.05, 0.95, n_chord)
    y_positions = np.linspace(-0.5 * span, 0.5 * span, n_span)

    rows = []
    node_id = 0
    for i, y in enumerate(y_positions):
        span_eta = abs(y) / (0.5 * span + 1e-9)
        span_lift_shape = 1.0 - 0.75 * span_eta**1.3

        for j, x_eta in enumerate(x_positions):
            z = naca_4digit_thickness(x_eta, t=0.11) * (1.0 - 0.3 * span_eta)

            gamma = (0.65 + 0.35 * np.sin(np.pi * x_eta)) * span_lift_shape * (1 + 0.03 * alpha_deg)
            cp = -1.6 * gamma + 0.35 + np.random.normal(0, 0.015)

            local_lift = q_inf * (0.007 + 0.022 * gamma + 0.0015 * alpha_deg)
            local_drag = q_inf * (0.0018 + 0.0012 * abs(cp) + 0.00065 * x_eta**2 + 0.0003 * span_eta)

            rows.append({
                'node_id': node_id,
                'span_index': i,
                'chord_index': j,
                'x': float(x_eta * chord),
                'y': float(y),
                'z': float(z),
                'gamma': float(gamma),
                'cp': float(cp),
                'lift': float(local_lift),
                'drag': float(local_drag),
            })
            node_id += 1

    df = pd.DataFrame(rows)
    return df, q_inf

In [ ]:
COLUMN_ALIASES = {
    'node_id': ['node_id', 'id', 'panel_id'],
    'span_index': ['span_index', 'span_idx', 'i_span'],
    'chord_index': ['chord_index', 'chord_idx', 'i_chord'],
    'x': ['x', 'x_cp', 'x_coord', 'x_position'],
    'y': ['y', 'y_cp', 'y_coord', 'y_position'],
    'z': ['z', 'z_cp', 'z_coord', 'z_position'],
    'lift': ['lift', 'local_lift', 'fz', 'force_z'],
    'drag': ['drag', 'local_drag', 'fx', 'force_x'],
    'cp': ['cp', 'pressure_coefficient', 'c_p'],
    'gamma': ['gamma', 'circulation', 'vortex_strength'],
}

def _read_tabular_file(path):
    suffix = path.suffix.lower()
    if suffix in ['.csv', '.txt']:
        return pd.read_csv(path)
    if suffix in ['.tsv']:
        return pd.read_csv(path, sep='\t')
    if suffix in ['.parquet']:
        return pd.read_parquet(path)
    if suffix in ['.json']:
        return pd.read_json(path)
    return None

def _normalize_real_nodes(df, q_inf, n_span=12, n_chord=18):
    work = df.copy()

    rename_map = {}
    for target, aliases in COLUMN_ALIASES.items():
        for alias in aliases:
            if alias in work.columns:
                rename_map[alias] = target
                break
    work = work.rename(columns=rename_map)

    if 'x' not in work.columns or 'y' not in work.columns:
        return None

    numeric_cols = ['x', 'y', 'z', 'lift', 'drag', 'cp', 'gamma']
    for col in numeric_cols:
        if col in work.columns:
            work[col] = pd.to_numeric(work[col], errors='coerce')

    work = work.dropna(subset=['x', 'y']).copy()
    if len(work) < 12:
        return None

    if 'z' not in work.columns:
        work['z'] = 0.0

    if 'node_id' not in work.columns:
        work['node_id'] = np.arange(len(work), dtype=int)

    if 'span_index' not in work.columns:
        work['span_index'] = pd.qcut(
            work['y'].rank(method='first'),
            q=min(n_span, len(work)),
            labels=False,
            duplicates='drop'
        ).fillna(0).astype(int)

    if 'chord_index' not in work.columns:
        work['chord_index'] = pd.qcut(
            work['x'].rank(method='first'),
            q=min(n_chord, len(work)),
            labels=False,
            duplicates='drop'
        ).fillna(0).astype(int)

    x_min, x_max = work['x'].min(), work['x'].max()
    x_scale = max(x_max - x_min, 1e-6)
    x_norm = (work['x'] - x_min) / x_scale

    if 'cp' not in work.columns or work['cp'].isna().all():
        work['cp'] = -0.9 - 0.5 * x_norm

    if 'gamma' not in work.columns or work['gamma'].isna().all():
        work['gamma'] = np.maximum(0.05, 0.75 - 0.35 * x_norm)

    if 'lift' not in work.columns or work['lift'].isna().all():
        work['lift'] = q_inf * (0.006 + 0.020 * work['gamma'])

    if 'drag' not in work.columns or work['drag'].isna().all():
        work['drag'] = q_inf * (0.0015 + 0.0011 * np.abs(work['cp']) + 0.00045 * x_norm**2)

    schema_cols = ['node_id', 'span_index', 'chord_index', 'x', 'y', 'z', 'lift', 'drag', 'cp', 'gamma']
    out = work[schema_cols].copy()
    out = out.replace([np.inf, -np.inf], np.nan).dropna()

    out['node_id'] = out['node_id'].astype(int)
    out['span_index'] = out['span_index'].astype(int)
    out['chord_index'] = out['chord_index'].astype(int)

    if len(out) < 12:
        return None

    return out.reset_index(drop=True)

def discover_candidate_files(base_dirs=None):
    if base_dirs is None:
        base_dirs = [Path('data'), Path('notebooks/data'), Path('data/training-datasets')]

    patterns = [
        '**/*naca*.csv', '**/*nasa*.csv', '**/*airfoil*.csv',
        '**/*naca*.parquet', '**/*nasa*.parquet', '**/*airfoil*.parquet',
        '**/*naca*.json', '**/*nasa*.json', '**/*airfoil*.json'
    ]

    found = []
    for base in base_dirs:
        if not base.exists():
            continue
        for pattern in patterns:
            found.extend(base.glob(pattern))

    unique = []
    seen = set()
    for path in found:
        key = str(path.resolve())
        if key not in seen:
            seen.add(key)
            unique.append(path)

    return unique

def load_nodes_with_adapter(preferred_path=None, synthetic_kwargs=None):
    synthetic_kwargs = synthetic_kwargs or {}
    default_q_inf = 0.5 * 1.225 * 72.0**2

    candidates = []
    if preferred_path:
        preferred = Path(preferred_path)
        if preferred.exists():
            candidates.append(preferred)

    candidates.extend(discover_candidate_files())

    for path in candidates:
        try:
            raw = _read_tabular_file(path)
            if raw is None or raw.empty:
                continue

            q_inf = default_q_inf
            if 'q_inf' in raw.columns:
                q_values = pd.to_numeric(raw['q_inf'], errors='coerce').dropna()
                if len(q_values) > 0:
                    q_inf = float(q_values.iloc[0])

            normalized = _normalize_real_nodes(raw, q_inf=q_inf)
            if normalized is not None and len(normalized) >= 12:
                info = {
                    'source': 'real_dataset',
                    'path': str(path),
                    'rows': int(len(normalized))
                }
                return normalized, q_inf, info
        except Exception as exc:
            print(f'Skipping dataset {path}: {exc}')

    synthetic_nodes, q_inf = generate_vlm_nodes(**synthetic_kwargs)
    info = {
        'source': 'synthetic_fallback',
        'path': None,
        'rows': int(len(synthetic_nodes))
    }
    return synthetic_nodes, q_inf, info

In [ ]:
# Adapter usage: change preferred_path to point at a specific NASA/NACA file if desired
nodes, q_inf, source_info = load_nodes_with_adapter(preferred_path=None)
nodes['score'] = nodes['lift'] - nodes['drag']

print('Dataset source:', source_info['source'])
print('Dataset path:', source_info['path'])
print('Rows:', source_info['rows'])
nodes.head()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

p0 = ax[0].scatter(nodes['y'], nodes['x'], c=nodes['lift'], cmap='viridis', s=28)
ax[0].set_title('VLM Nodes: Local Lift')
ax[0].set_xlabel('Span y [m]')
ax[0].set_ylabel('Chord x [m]')
plt.colorbar(p0, ax=ax[0], label='Lift [N]')

p1 = ax[1].scatter(nodes['y'], nodes['x'], c=nodes['drag'], cmap='magma', s=28)
ax[1].set_title('VLM Nodes: Local Drag')
ax[1].set_xlabel('Span y [m]')
ax[1].set_ylabel('Chord x [m]')
plt.colorbar(p1, ax=ax[1], label='Drag [N]')

plt.tight_layout()
plt.show()

In [ ]:
# QUBO: minimize drag - lift + smoothness coupling
N = min(28, len(nodes))
subset = nodes.iloc[:N].copy()
lift_norm = subset['lift'] / max(subset['lift'].max(), 1e-9)
drag_norm = subset['drag'] / max(subset['drag'].max(), 1e-9)

Q = np.zeros((N, N))
for i in range(N):
    Q[i, i] = 1.0 * drag_norm.iloc[i] - 1.0 * lift_norm.iloc[i]

for i in range(N):
    for j in range(i + 1, N):
        pi = subset.iloc[i][['x', 'y', 'z']].to_numpy()
        pj = subset.iloc[j][['x', 'y', 'z']].to_numpy()
        d = np.linalg.norm(pi - pj)
        coupling = 0.015 / (1.0 + d)
        Q[i, j] = coupling
        Q[j, i] = coupling

def qubo_cost(x, Q):
    return float(x @ Q @ x)

# Classical greedy fallback solver
x = np.zeros(N, dtype=int)
improved = True
while improved:
    improved = False
    current = qubo_cost(x, Q)
    for k in range(N):
        x_try = x.copy()
        x_try[k] = 1 - x_try[k]
        c_try = qubo_cost(x_try, Q)
        if c_try < current:
            x = x_try
            current = c_try
            improved = True

selected = subset[x == 1].copy()
print('Selected nodes:', len(selected), '/', N)
print('QUBO cost:', qubo_cost(x, Q))
selected[['node_id', 'lift', 'drag', 'score']].head()

In [ ]:
# Optional quantum path with Qiskit (falls back silently if unavailable)
quantum_solution = None
try:
    from qiskit_optimization import QuadraticProgram
    from qiskit_optimization.algorithms import MinimumEigenOptimizer
    from qiskit_algorithms import NumPyMinimumEigensolver

    qp = QuadraticProgram()
    for i in range(N):
        qp.binary_var(name=f'x{i}')

    linear = {f'x{i}': float(Q[i, i]) for i in range(N)}
    quadratic = {}
    for i in range(N):
        for j in range(i + 1, N):
            quadratic[(f'x{i}', f'x{j}')] = float(Q[i, j])

    qp.minimize(linear=linear, quadratic=quadratic)
    solver = MinimumEigenOptimizer(NumPyMinimumEigensolver())
    result = solver.solve(qp)
    quantum_solution = np.array([int(result.variables_dict[f'x{i}']) for i in range(N)])
    print('Quantum-like selected nodes:', quantum_solution.sum())
except Exception as exc:
    print('Quantum stack unavailable, using classical fallback only:', exc)

In [ ]:
# Coupled CFD proxy loop
solution = quantum_solution if quantum_solution is not None else x
selected_ratio = float(solution.sum()) / max(len(solution), 1)

x_span = max(nodes['y'].max() - nodes['y'].min(), 1e-6)
x_chord = max(nodes['x'].max() - nodes['x'].min(), 1e-6)
S_ref = x_span * x_chord
baseline_cl = nodes['lift'].sum() / (q_inf * S_ref)
baseline_cd = nodes['drag'].sum() / (q_inf * S_ref)

history = []
cl, cd = baseline_cl, baseline_cd
for it in range(1, 8):
    lift_boost = 1 + min(0.12, 0.03 + 0.11 * selected_ratio)
    drag_reduction = 1 - min(0.10, 0.02 + 0.09 * selected_ratio)

    cl_q = cl * lift_boost
    cd_q = max(cd * drag_reduction, 1e-6)

    yaw_penalty = 1 + 0.004 * it
    cl = cl_q * (0.996 - 0.001 * it)
    cd = cd_q * yaw_penalty

    history.append({
        'iteration': it,
        'cl': cl,
        'cd': cd,
        'l_over_d': cl / max(cd, 1e-6),
        'selected_ratio': selected_ratio,
    })

hist = pd.DataFrame(history)
hist

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4))

ax[0].plot(hist['iteration'], hist['cl'], label='CL', marker='o')
ax[0].plot(hist['iteration'], hist['cd'], label='CD', marker='s')
ax[0].set_title('Coupled Iteration Coefficients')
ax[0].set_xlabel('Iteration')
ax[0].legend()

ax[1].plot(hist['iteration'], hist['l_over_d'], color='tab:green', marker='^')
ax[1].set_title('L/D Trend in Coupled Loop')
ax[1].set_xlabel('Iteration')

plt.tight_layout()
plt.show()

print('Baseline CL, CD:', round(baseline_cl, 4), round(baseline_cd, 4))
print('Final coupled CL, CD:', round(hist.iloc[-1]['cl'], 4), round(hist.iloc[-1]['cd'], 4))

## Next Integration Step

Use the same payload shape against the backend endpoints:
- `POST /api/physics/vlm/solve`
- `POST /api/quantum/optimize-vlm-nodes`
- `POST /api/simulation/run`

This notebook is now adapter-ready: it consumes real NASA/NACA tabular files when present and falls back to synthetic generation when not available.